# Know Your Micro-Watershed

Read the area, basin fields, elevation, terrain and drainage of one MWS, then map its upstream and downstream links.

Run the cells in order. Each step uses data from the previous cells. You can edit the place, identifier, columns and chart settings as you go.


## Set up Python

Run these two collapsed cells once. They load the libraries and starting location. Expand them to see or change the setup.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import ast
import json
from getpass import getpass
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
GEOSERVER = 'https://geoserver.core-stack.org:8443/geoserver/'
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


## Choose the tehsil

The starting place is Hilsa, Nalanda, Bihar. A notebook downloaded from GeoLibre uses the selected tehsil. Change these names to read another place.


In [ ]:
state = SCOPE["state"].lower().replace(" ", "_")
district = SCOPE["district"].lower().replace(" ", "_")
tehsil = SCOPE["tehsil"].lower().replace(" ", "_")
place = {"state": state, "district": district, "tehsil": tehsil}
place


## The layers we will use

Each link reads a vector layer as GeoJSON. GeoJSON contains a feature list; each feature has a shape and its data fields. Village and waterbody intersection tables will be made from these vector shapes.


In [ ]:
layer_urls = {
    "mws": f"{GEOSERVER}mws/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws:mws_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "connectivity": f"{GEOSERVER}mws_connectivity/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws_connectivity:{district}_{tehsil}_mws_connectivity&outputFormat=application/json&srsName=EPSG:4326",
    "terrain": f"{GEOSERVER}terrain/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=terrain:{district}_{tehsil}_cluster&outputFormat=application/json&srsName=EPSG:4326",
    "elevation": f"{GEOSERVER}dem/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=dem:{district}_{tehsil}_dem_vector&outputFormat=application/json&srsName=EPSG:4326",
    "rivers": f"{GEOSERVER}river/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=river:{district}_{tehsil}_river_vector&outputFormat=application/json&srsName=EPSG:4326",
    "canals": f"{GEOSERVER}canal/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=canal:{district}_{tehsil}_canal_vector&outputFormat=application/json&srsName=EPSG:4326",
    "drainage": f"{GEOSERVER}drainage_density/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=drainage_density:{district}_{tehsil}_drainage_density&outputFormat=application/json&srsName=EPSG:4326",
    "stream_order": f"{GEOSERVER}stream_order/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=stream_order:stream_order_{district}_{tehsil}_vector&outputFormat=application/json&srsName=EPSG:4326",
    "villages": f"{GEOSERVER}panchayat_boundaries/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=panchayat_boundaries:{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "waterbodies": f"{GEOSERVER}swb/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=swb:surface_waterbodies_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
}
pd.DataFrame(layer_urls.items(), columns=["Layer", "GeoJSON URL"])


## Read the MWS boundaries

Read the shapes and their published fields.


In [ ]:
mws_response = requests.get(layer_urls["mws"], timeout=90)
mws_response.raise_for_status()
mws_geojson = mws_response.json()
mws = gpd.GeoDataFrame.from_features(mws_geojson["features"], crs="EPSG:4326")
mws.drop(columns="geometry").head()


## Choose one micro-watershed

`uid` is the MWS identifier in this layer. Start with the first one, or replace `mws_id` with another identifier from the displayed list.


In [ ]:
mws_ids = mws["uid"].sort_values().tolist()
display(pd.DataFrame({"MWS identifier": mws_ids}))
mws_id = mws_ids[0]
mws_id


## Area and basin details

Select the MWS by its exact `uid`. The basin fields are `wsconc`, `bacode` and `sbcode`. Fields without a recorded value remain blank.


In [ ]:
selected_mws = mws.loc[mws["uid"] == mws_id]
selected_mws.reindex(columns=["uid", "area_in_ha", "wsconc", "bacode", "sbcode"]).rename(columns={
    "area_in_ha": "Area (ha)", "wsconc": "Watershed code", "bacode": "Basin code", "sbcode": "Sub-basin code"
}).T


## Read the elevation summary

This vector layer gives the minimum, maximum and mean elevation for each MWS.


In [ ]:
elevation_response = requests.get(layer_urls["elevation"], timeout=90)
elevation_response.raise_for_status()
elevation_geojson = elevation_response.json()
elevation = gpd.GeoDataFrame.from_features(elevation_geojson["features"], crs="EPSG:4326")
elevation.drop(columns="geometry").head()


## Elevation in metres

Read the three elevation columns for the selected MWS.


In [ ]:
elevation.loc[elevation["uid"] == mws_id, ["min_elevation", "max_elevation", "mean_elevation"]].rename(columns={
    "min_elevation": "Minimum (m)", "max_elevation": "Maximum (m)", "mean_elevation": "Mean (m)"
}).T


## Read the terrain layer

The terrain columns describe shares of the MWS area.


In [ ]:
terrain_response = requests.get(layer_urls["terrain"], timeout=90)
terrain_response.raise_for_status()
terrain_geojson = terrain_response.json()
terrain = gpd.GeoDataFrame.from_features(terrain_geojson["features"], crs="EPSG:4326")
terrain.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_terrain_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['uid', 'plain_area', 'hill_slope', 'ridge_area', 'slopy_area', 'valley_are']), ["name", "type", "description"]]


## Terrain shares

These five fields are already percentages. They do not need to be divided by the MWS area.


In [ ]:
terrain_columns = {"plain_area": "Plains (%)", "slopy_area": "Sloping land (%)", "hill_slope": "Hill slopes (%)",
                   "ridge_area": "Ridges (%)", "valley_are": "Valleys (%)"}
terrain.loc[terrain["uid"] == mws_id, list(terrain_columns)].rename(columns=terrain_columns).T


## Read MWS connections

`upstream` contains a list stored as text. `downstream` contains an identifier, or an empty string where no link is recorded.


In [ ]:
connectivity_response = requests.get(layer_urls["connectivity"], timeout=90)
connectivity_response.raise_for_status()
connectivity_geojson = connectivity_response.json()
connectivity = gpd.GeoDataFrame.from_features(connectivity_geojson["features"], crs="EPSG:4326")
connectivity.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_mws_connectivity_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['uid', 'upstream', 'downstream', 'direction']), ["name", "type", "description"]]


## List upstream and downstream MWS

Use `ast.literal_eval` to read the upstream list. Keep the downstream identifier as a string, including its underscore.


In [ ]:
connection = connectivity.loc[connectivity["uid"] == mws_id].iloc[0]
upstream_ids = ast.literal_eval(connection["upstream"])
downstream_ids = [connection["downstream"]] if connection["downstream"] else []
display(pd.DataFrame({"Upstream MWS": upstream_ids}))
display(pd.DataFrame({"Downstream MWS": downstream_ids}))


## Map the connections

Blue MWS contribute water to the selected MWS. Orange MWS receive it. Only shapes in the loaded tehsil layer can be drawn.


In [ ]:
upstream_mws = mws.loc[mws["uid"].isin(upstream_ids)]
downstream_mws = mws.loc[mws["uid"].isin(downstream_ids)]
ax = selected_mws.plot(color="grey", edgecolor="black", figsize=(7, 6))
if not upstream_mws.empty:
    upstream_mws.plot(ax=ax, color="steelblue", edgecolor="white")
if not downstream_mws.empty:
    downstream_mws.plot(ax=ax, color="darkorange", edgecolor="white")
ax.set(title="Selected: grey · upstream: blue · downstream: orange", xlabel="Longitude", ylabel="Latitude")
plt.show()


## Read drainage density

Drainage density describes stream length relative to area.


In [ ]:
drainage_response = requests.get(layer_urls["drainage"], timeout=90)
drainage_response.raise_for_status()
drainage_geojson = drainage_response.json()
drainage = gpd.GeoDataFrame.from_features(drainage_geojson["features"], crs="EPSG:4326")
drainage.drop(columns="geometry").head()


## Drainage-density values

Read the published weighted value and standard deviation in km/km².


In [ ]:
drainage.loc[drainage["uid"] == mws_id, ["drainage_density_weighted", "drainage_density_std"]].rename(columns={
    "drainage_density_weighted": "Weighted drainage density (km/km²)",
    "drainage_density_std": "Standard deviation (km/km²)"
}).T


## Read stream-order shares

Columns `1` through `11` give the published area shares for the stream orders.


In [ ]:
stream_order_response = requests.get(layer_urls["stream_order"], timeout=90)
stream_order_response.raise_for_status()
stream_order_geojson = stream_order_response.json()
stream_order = gpd.GeoDataFrame.from_features(stream_order_geojson["features"], crs="EPSG:4326")
stream_order.drop(columns="geometry").head()


## Stream-order shares

Turn the selected row into a short table and bar chart.


In [ ]:
order_columns = [str(order) for order in range(1, 12)]
order_shares = stream_order.loc[stream_order["uid"] == mws_id, order_columns].iloc[0]
display(order_shares.rename_axis("Stream order").to_frame("Area share (%)"))
order_shares.plot.bar(figsize=(8, 3), xlabel="Stream order", ylabel="Area share (%)")
plt.show()


## Read the rivers

Inspect the river features provided for the tehsil.


In [ ]:
rivers_response = requests.get(layer_urls["rivers"], timeout=90)
rivers_response.raise_for_status()
rivers_geojson = rivers_response.json()
rivers = gpd.GeoDataFrame.from_features(rivers_geojson["features"], crs="EPSG:4326")
rivers.drop(columns="geometry").head()


## Read the canals

Inspect the canal features provided for the tehsil.


In [ ]:
canals_response = requests.get(layer_urls["canals"], timeout=90)
canals_response.raise_for_status()
canals_geojson = canals_response.json()
canals = gpd.GeoDataFrame.from_features(canals_geojson["features"], crs="EPSG:4326")
canals.drop(columns="geometry").head()


## Read the village boundaries

The census boundary layer uses `vill_ID` and `vill_name`.


In [ ]:
villages_response = requests.get(layer_urls["villages"], timeout=90)
villages_response.raise_for_status()
villages_geojson = villages_response.json()
villages = gpd.GeoDataFrame.from_features(villages_geojson["features"], crs="EPSG:4326")
villages.drop(columns="geometry").head()


## Villages intersecting this MWS

Use the selected MWS shape to find intersecting village shapes. A village can intersect more than one MWS.


In [ ]:
mws_shape = selected_mws.geometry.iloc[0]
mws_intersect_villages = villages.loc[villages.intersects(mws_shape), ["vill_ID", "vill_name"]]
mws_intersect_villages


## Read the surface waterbodies

The surface-waterbody identifier is `UID`, with capital letters.


In [ ]:
waterbodies_response = requests.get(layer_urls["waterbodies"], timeout=90)
waterbodies_response.raise_for_status()
waterbodies_geojson = waterbodies_response.json()
waterbodies = gpd.GeoDataFrame.from_features(waterbodies_geojson["features"], crs="EPSG:4326")
waterbodies.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_surface_water_bodies_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['UID', 'area_ored', 'area_17-18', 'k_17-18', 'kr_17-18', 'krz_17-18']), ["name", "type", "description"]]


## Waterbodies intersecting this MWS

The result lists waterbody identifiers; it does not divide waterbody area between MWS.


In [ ]:
mws_intersect_swb = waterbodies.loc[waterbodies.intersects(mws_shape), ["UID"]]
mws_intersect_swb


## Connect to the CoRE Stack API

The [API guide](https://api-doc.core-stack.org) explains access. The key goes in the `X-API-Key` header. This cell reads `CORE_STACK_API_KEY` from your environment, or asks for it without showing it. The key is not written into the notebook.


In [ ]:
api_key = os.environ.get("CORE_STACK_API_KEY") or getpass("CoRE Stack API key: ")
api_headers = {"X-API-Key": api_key}


## Read the tehsil tables from the API

`get_tehsil_data` returns a dictionary of tables for the same place. Here we make the request once and reuse the returned tables below.


In [ ]:
api_response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_response.raise_for_status()
api_data = api_response.json()
pd.DataFrame({"Table": api_data.keys(), "Rows": [len(rows) for rows in api_data.values()]})


## Area and basin details from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_mws = pd.DataFrame(api_data['mws'])
api_mws = api_mws.loc[api_mws['uid'] == mws_id]
api_mws.reindex(columns=['uid', 'area_in_ha', 'watershed_code', 'basin_code', 'sub_basin_code']).T


## Elevation from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_elevation = pd.DataFrame(api_data['dem'])
api_elevation = api_elevation.loc[api_elevation['uid'] == mws_id]
api_elevation.reindex(columns=['min_elevation_in_m', 'max_elevation_in_m', 'mean_elevation_in_m']).T


## Terrain from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_terrain = pd.DataFrame(api_data['terrain'])
api_terrain = api_terrain.loc[api_terrain['uid'] == mws_id]
api_terrain.reindex(columns=['plain_area_percent', 'slopy_area_percent', 'hill_slope_area_percent', 'ridge_area_percent', 'valley_area_percent']).T


## MWS connections from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_connections = pd.DataFrame(api_data['mws_connectivity'])
api_connections = api_connections.loc[api_connections['uid'] == mws_id]
api_connections.reindex(columns=['upstream_mws', 'downstream_mws', 'direction']).T


## Drainage density from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_drainage = pd.DataFrame(api_data['drainage_density'])
api_drainage = api_drainage.loc[api_drainage['uid'] == mws_id]
api_drainage.reindex(columns=['drainage_density_weighted_in_km_per_km2', 'drainage_density_std_in_km_per_km2']).T


## Stream-order shares from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_orders = pd.DataFrame(api_data['stream_order'])
api_orders = api_orders.loc[api_orders['uid'] == mws_id]
api_orders.reindex(columns=['order_1_area_percent', 'order_2_area_percent', 'order_3_area_percent', 'order_4_area_percent', 'order_5_area_percent', 'order_6_area_percent', 'order_7_area_percent', 'order_8_area_percent', 'order_9_area_percent', 'order_10_area_percent', 'order_11_area_percent']).T


## Village intersections from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_villages = pd.DataFrame(api_data['mws_intersect_villages'])
api_villages = api_villages.loc[api_villages['mws uid'] == mws_id]
api_villages.T


## Waterbody intersections from the API

Read the API table for the same MWS. The column names identify the units.


In [ ]:
api_waterbodies = pd.DataFrame(api_data['mws_intersect_swb'])
api_waterbodies = api_waterbodies.loc[api_waterbodies['uid'] == mws_id]
api_waterbodies.T
